# ChatOllama


> 업데이트 기준: **2026-09-18**  
> 책의 학습 목표는 유지하면서 LangChain 1.x의 분리된 provider 패키지와 현재 메시지/스트리밍 API에 맞췄습니다. 모델 이름은 공급자 정책에 따라 바뀔 수 있으므로 환경 변수로 덮어쓸 수 있게 구성했습니다.


Ollama를 설치하고 서버를 시작한 뒤 모델을 받습니다.

```bash
ollama pull gpt-oss:20b
ollama pull gemma3:4b   # 이미지 예제용
ollama list
```

메모리가 부족하면 `OLLAMA_MODEL` 환경 변수에 더 작은 모델 태그를 지정하세요.


In [ ]:
%pip install -qU langchain-ollama pillow pydantic python-dotenv


In [ ]:
import os
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_ollama import ChatOllama

load_dotenv()
model_name = os.getenv("OLLAMA_MODEL", "gpt-oss:20b")
llm = ChatOllama(
    model=model_name,
    temperature=0,
    validate_model_on_init=True,
)

prompt = ChatPromptTemplate.from_template("{topic}을 두 문장으로 설명해 주세요.")
chain = prompt | llm | StrOutputParser()

for chunk in chain.stream({"topic": "딥러닝"}):
    print(chunk, end="", flush=True)


## 비동기 스트리밍


In [ ]:
async for chunk in chain.astream({"topic": "검색 증강 생성"}):
    print(chunk, end="", flush=True)


## 구조화 출력

문자열에 ‘JSON으로 답하라’고만 쓰는 대신 Pydantic 스키마로 결과를 검증합니다. 선택한 Ollama 모델이 structured output을 지원해야 합니다.


In [ ]:
from pydantic import BaseModel, Field


class TravelPlan(BaseModel):
    places: list[str] = Field(description="유럽 여행지 이름 10개")


structured_llm = llm.with_structured_output(TravelPlan)
plan = structured_llm.invoke("유럽 여행지 10곳을 추천해 주세요.")
print(plan)


## 멀티모달 이미지 입력

현재 공식 예시의 content block 형식을 사용합니다. `gemma3`처럼 이미지 입력을 지원하는 모델이 필요합니다.


In [ ]:
import base64
from pathlib import Path
from langchain.messages import HumanMessage

image_path = Path("images/jeju-beach.jpg")
if not image_path.exists():
    raise FileNotFoundError(f"샘플 이미지를 준비하세요: {image_path.resolve()}")

image_b64 = base64.b64encode(image_path.read_bytes()).decode("utf-8")
vision_llm = ChatOllama(
    model=os.getenv("OLLAMA_VISION_MODEL", "gemma3:4b"),
    temperature=0,
    validate_model_on_init=True,
)
message = HumanMessage(
    content=[
        {"type": "text", "text": "이미지를 세 개의 불릿으로 설명해 주세요."},
        {
            "type": "image_url",
            "image_url": f"data:image/jpeg;base64,{image_b64}",
        },
    ]
)
print(vision_llm.invoke([message]).text)
